# LeetCode 587 — Erect the Fence bằng Python

- Link: https://leetcode.com/problems/erect-the-fence/
- Độ khó: **Hard**
- Chủ đề: Hình học tính toán, Bao lồi (Convex Hull), Tích có hướng (cross product), Stack, Sắp xếp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def outerTrees(self, trees: List[List[int]]) -> List[List[int]]:

```

## 1. Giới thiệu & mục tiêu học

Bài này là **bao lồi (convex hull)** — thuật toán hình học kinh điển nhất. Điểm "Hard" của LeetCode 587 là phải trả về **cả những điểm nằm trên cạnh** của bao lồi (điểm thẳng hàng), điều mà phần lớn cài đặt sách giáo khoa bỏ qua.

Sau notebook này bạn sẽ:

1. Hiểu và dùng thành thạo **tích có hướng (cross product)** để biết ba điểm "rẽ trái", "rẽ phải" hay "thẳng hàng".
2. Viết lời giải vét cạn `O(n^3)` dựa trên định nghĩa "đường thẳng tựa".
3. Cài đặt ba thuật toán bao lồi: **Gói quà (Jarvis march)**, **Graham scan**, **Monotone chain (Andrew)** — đều giữ được điểm thẳng hàng.
4. Nắm các trường hợp biên: 1 điểm, 2 điểm, mọi điểm thẳng hàng, nhiều điểm trên một cạnh.
5. Kiểm chứng bằng brute force trên lưới nhỏ (nơi điểm thẳng hàng xuất hiện rất nhiều).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `trees`, `trees[i] = [x_i, y_i]` là tọa độ một cây trong vườn. Cần quây hàng rào bằng **sợi dây ngắn nhất** bao quanh **mọi** cây. Trả về tọa độ của **các cây nằm đúng trên hàng rào** (kể cả cây nằm giữa một đoạn rào thẳng). Thứ tự trả về tùy ý.

"Sợi dây ngắn nhất bao mọi điểm" chính là **biên của bao lồi**.

**Ví dụ 1**

```text
Input:  trees = [[1,1],[2,2],[2,0],[2,4],[3,3],[4,2]]
Output: [[1,1],[2,0],[4,2],[3,3],[2,4]]
Giải thích: chỉ cây [2,2] nằm bên trong hàng rào. Cây [3,3] nằm trên đoạn rào nối [4,2] và [2,4].
```

**Ví dụ 2**

```text
Input:  trees = [[1,2],[2,2],[4,2]]
Output: [[4,2],[2,2],[1,2]]
Giải thích: ba cây thẳng hàng — hàng rào là một đoạn thẳng, cả ba cây đều nằm trên đó.
```

**Ràng buộc**

- `1 <= trees.length <= 3000`
- `trees[i].length == 2`
- `0 <= x_i, y_i <= 100`
- Mọi vị trí là **khác nhau**.

In [1]:
trees1 = [[1, 1], [2, 2], [2, 0], [2, 4], [3, 3], [4, 2]]
trees2 = [[1, 2], [2, 2], [4, 2]]

def draw(points, highlight=()):
    """Vẽ lưới ASCII: '#' = cây trên hàng rào, 'o' = cây bên trong, '.' = trống."""
    hs = {tuple(p) for p in highlight}
    pts = {tuple(p) for p in points}
    W = max(x for x, _ in pts) + 1
    H = max(y for _, y in pts) + 1
    for y in range(H - 1, -1, -1):
        row = ""
        for x in range(W):
            row += "#" if (x, y) in hs else ("o" if (x, y) in pts else ".")
        print(f"{y} {row}")
    print("  " + "".join(str(x % 10) for x in range(W)))

draw(trees1, [[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])

4 ..#..
3 ...#.
2 ..o.#
1 .#...
0 ..#..
  01234


## 3. Kiến thức nền

### 3.1. Tích có hướng (cross product) — công cụ số 1

Với ba điểm `O, A, B`:

```text
cross(O, A, B) = (A.x - O.x) * (B.y - O.y) - (A.y - O.y) * (B.x - O.x)
```

- `> 0`: đi `O → A → B` là **rẽ trái** (ngược chiều kim đồng hồ). `B` nằm **bên trái** tia `O→A`.
- `< 0`: **rẽ phải** (thuận chiều kim đồng hồ). `B` nằm bên phải.
- `= 0`: **thẳng hàng**.

Chỉ dùng phép nhân, trừ số nguyên → **không sai số** (khác với dùng góc `atan2`).

In [2]:
def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

O, A = (0, 0), (2, 0)
for B in [(1, 1), (1, -1), (5, 0), (-3, 0)]:
    c = cross(O, A, B)
    kind = "rẽ trái" if c > 0 else ("rẽ phải" if c < 0 else "thẳng hàng")
    print(f"O={O}, A={A}, B={B}: cross = {c:>3} -> {kind}")

O=(0, 0), A=(2, 0), B=(1, 1): cross =   2 -> rẽ trái
O=(0, 0), A=(2, 0), B=(1, -1): cross =  -2 -> rẽ phải
O=(0, 0), A=(2, 0), B=(5, 0): cross =   0 -> thẳng hàng
O=(0, 0), A=(2, 0), B=(-3, 0): cross =   0 -> thẳng hàng


### 3.2. Điểm nằm trên đoạn thẳng

`R` nằm trên đoạn `PQ` ⇔ `cross(P, Q, R) = 0` (thẳng hàng) **và** `R` nằm trong hình chữ nhật bao `P, Q` (`min(Px,Qx) <= Rx <= max(Px,Qx)`, tương tự với `y`).

In [3]:
def on_segment(p, q, r):
    return (cross(p, q, r) == 0
            and min(p[0], q[0]) <= r[0] <= max(p[0], q[0])
            and min(p[1], q[1]) <= r[1] <= max(p[1], q[1]))

print(on_segment((0, 0), (4, 4), (2, 2)), on_segment((0, 0), (4, 4), (5, 5)), on_segment((0, 0), (4, 4), (2, 3)))

True False False


### 3.3. Sắp xếp tuple và sắp xếp với `functools.cmp_to_key`

- `sorted(points)` với điểm là tuple: sắp theo `x` rồi `y` — dùng cho Monotone chain.
- Graham scan cần sắp theo **góc** quanh một điểm. Ta viết hàm so sánh `cmp(a, b)` trả số âm/0/dương và bọc bằng `cmp_to_key`.

In [4]:
from functools import cmp_to_key
pts = [(3, 1), (1, 2), (1, 0), (2, 2)]
print(sorted(pts))

def by_value_desc(a, b):
    return b[0] - a[0]        # âm -> a đứng trước
print(sorted(pts, key=cmp_to_key(by_value_desc)))

[(1, 0), (1, 2), (2, 2), (3, 1)]
[(3, 1), (2, 2), (1, 2), (1, 0)]


### 3.4. Stack bằng list

`append` để đẩy, `pop()` để lấy phần tử cuối, `st[-1]`, `st[-2]` là hai phần tử trên cùng. Cả Graham scan và Monotone chain đều là "stack bỏ điểm rẽ sai chiều".

In [5]:
st = []
for v in [1, 5, 3, 4]:
    while st and st[-1] > v:      # ví dụ: giữ stack tăng dần
        print(f"  pop {st.pop()} vì > {v}")
    st.append(v)
    print("stack:", st)

stack: [1]
stack: [1, 5]
  pop 5 vì > 3
stack: [1, 3]
stack: [1, 3, 4]


### 3.5. Khử trùng lặp bằng `set` của tuple

List không hash được nên không cho vào `set`. Đổi điểm sang `tuple` trước, cuối cùng đổi lại `list` để trả về.

In [6]:
hull = [(0, 0), (2, 0), (0, 0), (2, 2)]
print(sorted(set(hull)), [list(p) for p in sorted(set(hull))])

[(0, 0), (2, 0), (2, 2)] [[0, 0], [2, 0], [2, 2]]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Hình dung sợi dây chun

Cắm đinh ở mỗi cây, căng một sợi dây chun thật to rồi thả ra — nó co lại ôm sát các đinh ngoài cùng. Các đinh chạm dây là đáp án. Những đinh mà dây chạm **giữa một đoạn thẳng** cũng phải được tính.

### 4.2. Tính tay ví dụ 1 bằng tích có hướng

Đi ngược chiều kim đồng hồ quanh hàng rào: `(1,1) → (2,0) → (4,2) → (3,3) → (2,4) → (1,1)`.

- `cross((1,1), (2,0), (4,2)) = (1)(1) - (-1)(3) = 4 > 0` → rẽ trái ✓.
- `cross((2,0), (4,2), (3,3)) = (2)(3) - (2)(1) = 4 > 0` ✓.
- `cross((4,2), (3,3), (2,4)) = (-1)(2) - (1)(-2) = 0` → **thẳng hàng**: `(3,3)` nằm giữa cạnh `(4,2)–(2,4)`.
- Điểm `(2,2)` bên trong: với **mọi** cạnh `P→Q` của hàng rào (đi ngược chiều kim đồng hồ), `cross(P, Q, (2,2)) > 0` (nằm bên trái, tức phía trong).

### 4.3. Định nghĩa "đường thẳng tựa"

Đoạn `PQ` (với `P`, `Q` là hai cây) nằm trên hàng rào ⇔ **mọi cây đều nằm về một phía** của đường thẳng `PQ` (hoặc trên nó). Đây là định nghĩa dùng cho brute force.

In [7]:
hull_ccw = [(1, 1), (2, 0), (4, 2), (3, 3), (2, 4)]
for i in range(len(hull_ccw)):
    P, Q, R = hull_ccw[i], hull_ccw[(i + 1) % 5], hull_ccw[(i + 2) % 5]
    print(f"cross({P}, {Q}, {R}) = {cross(P, Q, R)}")
print("điểm (2,2) so với từng cạnh:", [cross(hull_ccw[i], hull_ccw[(i + 1) % 5], (2, 2)) for i in range(5)])

cross((1, 1), (2, 0), (4, 2)) = 4
cross((2, 0), (4, 2), (3, 3)) = 4
cross((4, 2), (3, 3), (2, 4)) = 0
cross((3, 3), (2, 4), (1, 1)) = 4
cross((2, 4), (1, 1), (2, 0)) = 4
điểm (2,2) so với từng cạnh: [2, 4, 2, 2, 2]


## 5. Cách 1 — Brute Force: thử mọi cặp điểm làm cạnh

**Ý tưởng.** Với mỗi cặp `(P, Q)`, kiểm tra mọi điểm `R` có cùng một phía của `PQ` không (tất cả `cross >= 0` hoặc tất cả `cross <= 0`). Nếu có, `PQ` nằm trên đường biên; mọi điểm nằm **trên đoạn** `PQ` đều thuộc hàng rào. Trường hợp chỉ có 1 cây: trả về chính nó.

In [8]:
def outer_trees_brute(trees):
    pts = [tuple(p) for p in trees]
    n = len(pts)
    if n == 1:
        return [list(pts[0])]
    on = set()
    for i in range(n):
        for j in range(i + 1, n):
            P, Q = pts[i], pts[j]
            signs = [cross(P, Q, R) for R in pts]
            if all(s >= 0 for s in signs) or all(s <= 0 for s in signs):
                for R in pts:
                    if on_segment(P, Q, R):
                        on.add(R)
    return [list(p) for p in on]

def norm(points):
    """Chuẩn hóa để so sánh: tập tuple đã sắp xếp."""
    return sorted(map(tuple, points))

print(norm(outer_trees_brute(trees1)))
print(norm(outer_trees_brute(trees2)))
assert norm(outer_trees_brute(trees1)) == norm([[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])
assert norm(outer_trees_brute(trees2)) == norm(trees2)

[(1, 1), (2, 0), (2, 4), (3, 3), (4, 2)]
[(1, 2), (2, 2), (4, 2)]


**Độ phức tạp:** `O(n^2)` cặp × `O(n)` kiểm tra → **`O(n^3)`**. Với `n = 3000` là `2.7 · 10^10` — quá chậm. Space `O(n)`.

**Nút thắt:** ta thử cả những cặp điểm nằm sâu bên trong. Nếu đi **dọc theo biên** — từ một đỉnh chắc chắn thuộc bao lồi sang đỉnh kế tiếp — thì không phải thử mọi cặp.

In [9]:
import time, random
random.seed(0)
for n in (40, 80, 120):
    pts = random.sample([(x, y) for x in range(60) for y in range(60)], n)
    t = time.perf_counter()
    outer_trees_brute(pts)
    print(f"n={n:>3}: {time.perf_counter() - t:.3f}s")

n= 40: 0.007s
n= 80: 0.049s


n=120: 0.123s


## 6. Bước đột phá

**Insight 1 — Có điểm chắc chắn nằm trên bao lồi.** Điểm có tọa độ nhỏ nhất theo thứ tự `(x, y)` (trái nhất, rồi thấp nhất) luôn là một **đỉnh** của bao lồi. Từ đó ta có thể "đi vòng" theo biên.

**Insight 2 — Đi ngược chiều kim đồng hồ thì luôn rẽ trái (hoặc đi thẳng).** Trên biên bao lồi theo chiều ngược kim đồng hồ, mọi bộ ba đỉnh liên tiếp có `cross >= 0`. Nếu gặp bộ ba **rẽ phải** (`cross < 0`), điểm giữa chắc chắn **không** nằm trên biên → bỏ nó. Đây là nguyên lý của Graham scan và Monotone chain.

**Quy tắc then chốt cho bài này (giữ điểm thẳng hàng):**
- **Chỉ pop khi `cross < 0`** (rẽ phải thật sự). Nếu pop cả khi `cross == 0` thì điểm nằm giữa cạnh bị mất → sai đề.
- Khi đi **gói quà**, trong các điểm thẳng hàng chọn điểm **xa nhất** làm đỉnh kế tiếp, rồi thêm mọi điểm trên đoạn đó.
- Khi **Graham** sắp theo góc, các điểm thẳng hàng trên **tia cuối cùng** phải được xếp **từ xa về gần** (để khi đi về điểm xuất phát chúng nằm đúng thứ tự trên biên).

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time |
|---|---|---|
| 1 | Brute force cặp điểm (mục 5) | `O(n^3)` |
| 2 | Gói quà (Jarvis march) | `O(n · h)`, `h` = số điểm trên biên |
| 3 | Graham scan | `O(n log n)` |
| 4 | Monotone chain (Andrew) | `O(n log n)` |

### 7.1. Cách 2 — Gói quà (Jarvis march)

**Giải thích.** Giống quấn giấy gói quà quanh hộp:

1. Bắt đầu từ `p` = điểm trái nhất (Insight 1).
2. Tìm `q` sao cho **không có điểm nào nằm bên phải** tia `p → q` (mọi điểm đều bên trái hoặc trên tia). Duyệt mọi `r`: nếu `r` nằm bên phải `p → q` (`cross < 0`) thì `q = r`. Nếu thẳng hàng thì lấy `r` **xa hơn**.
3. Thêm mọi điểm nằm trên đoạn `p → q` vào kết quả.
4. `p = q`; lặp tới khi quay về điểm đầu.

In [10]:
def dist2(a, b):
    return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

def outer_trees_jarvis(trees):
    pts = [tuple(p) for p in trees]
    n = len(pts)
    if n <= 3:
        return [list(p) for p in pts]          # <= 3 điểm: tất cả đều trên biên
    start = min(range(n), key=lambda i: pts[i])
    on = set()
    p = start
    while True:
        q = (p + 1) % n
        for r in range(n):
            c = cross(pts[p], pts[q], pts[r])
            if c < 0 or (c == 0 and dist2(pts[p], pts[r]) > dist2(pts[p], pts[q])):
                q = r                          # r "ngoài" hơn q
        for r in range(n):                     # mọi điểm trên cạnh p -> q
            if on_segment(pts[p], pts[q], pts[r]):
                on.add(pts[r])
        p = q
        if p == start:
            break
    return [list(x) for x in on]

print(norm(outer_trees_jarvis(trees1)))
assert norm(outer_trees_jarvis(trees1)) == norm(outer_trees_brute(trees1))

[(1, 1), (2, 0), (2, 4), (3, 3), (4, 2)]


**Đọc code:**

- `n <= 3`: ba điểm bất kỳ (tam giác hoặc thẳng hàng) đều nằm trên biên; trả luôn để tránh vòng lặp đặc biệt.
- `q = (p + 1) % n`: ứng viên ban đầu bất kỳ khác `p`.
- `c < 0`: `r` nằm bên phải `p → q`, tức `q` chưa phải "ngoài cùng" → thay bằng `r`.
- `c == 0` và `r` xa hơn: chọn điểm xa nhất trên cùng hướng, để cạnh đi tới **đỉnh** chứ không dừng ở điểm giữa.
- Vòng thứ hai thêm mọi điểm trên đoạn `p–q` (gồm cả hai đầu).

**Code nộp LeetCode (Cách 2):**

In [11]:
from typing import List

class SolutionJarvis:
    def outerTrees(self, trees: List[List[int]]) -> List[List[int]]:
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

        def d2(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        n = len(trees)
        if n <= 3:
            return trees
        start = min(range(n), key=lambda i: trees[i])
        on, p = set(), start
        while True:
            q = (p + 1) % n
            for r in range(n):
                c = cross(trees[p], trees[q], trees[r])
                if c < 0 or (c == 0 and d2(trees[p], trees[r]) > d2(trees[p], trees[q])):
                    q = r
            P, Q = trees[p], trees[q]
            for r in range(n):
                R = trees[r]
                if (cross(P, Q, R) == 0 and min(P[0], Q[0]) <= R[0] <= max(P[0], Q[0])
                        and min(P[1], Q[1]) <= R[1] <= max(P[1], Q[1])):
                    on.add(r)
            p = q
            if p == start:
                break
        return [trees[i] for i in on]

assert norm(SolutionJarvis().outerTrees(trees1)) == norm([[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])
assert norm(SolutionJarvis().outerTrees(trees2)) == norm(trees2)
print("SolutionJarvis OK")

SolutionJarvis OK


**Độ phức tạp Cách 2:** mỗi bước đi tốn `O(n)`, số bước = số đỉnh của bao lồi `h` → **`O(n · h)`**. Tốt khi bao lồi có ít đỉnh; tệ nhất `O(n^2)` (với tọa độ ≤ 100 thì `h` bị chặn khoảng vài trăm nên vẫn ổn).

### 7.2. Cách 3 — Graham scan

**Giải thích.**

1. Chọn `start` = điểm **thấp nhất** (rồi trái nhất).
2. Sắp các điểm còn lại theo **góc** quanh `start` (ngược chiều kim đồng hồ) bằng tích có hướng; cùng góc thì **gần trước, xa sau**.
3. **Sửa tia cuối:** các điểm thẳng hàng với `start` trên tia có góc lớn nhất phải **đảo thứ tự** (xa trước, gần sau), vì khi đi vòng về `start` ta gặp điểm xa trước.
4. Duyệt, dùng stack: trước khi đẩy `p`, **pop khi `cross(st[-2], st[-1], p) < 0`** (rẽ phải).

In [12]:
def outer_trees_graham(trees):
    pts = [tuple(p) for p in trees]
    n = len(pts)
    if n <= 3:
        return [list(p) for p in pts]
    start = min(pts, key=lambda p: (p[1], p[0]))
    rest = [p for p in pts if p != start]

    def cmp(a, b):
        c = cross(start, a, b)
        if c != 0:
            return -1 if c > 0 else 1                 # a có góc nhỏ hơn -> đứng trước
        return dist2(start, a) - dist2(start, b)      # cùng góc: gần trước

    rest.sort(key=cmp_to_key(cmp))
    # Đảo các điểm thẳng hàng trên tia cuối (xa trước, gần sau)
    i = len(rest) - 1
    while i > 0 and cross(start, rest[-1], rest[i - 1]) == 0:
        i -= 1
    rest[i:] = rest[i:][::-1]

    st = [start]
    for p in rest:
        while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
            st.pop()
        st.append(p)
    return [list(p) for p in st]

print(norm(outer_trees_graham(trees1)))
assert norm(outer_trees_graham(trees1)) == norm(outer_trees_brute(trees1))

[(1, 1), (2, 0), (2, 4), (3, 3), (4, 2)]


**Đọc code:**

- `cmp`: `cross(start, a, b) > 0` nghĩa là `b` nằm bên trái tia `start→a` → góc của `a` nhỏ hơn → `a` trước.
- Vòng `while i > 0 and cross(start, rest[-1], rest[i-1]) == 0`: tìm đoạn cuối các điểm cùng tia với điểm cuối.
- Nếu **mọi** điểm thẳng hàng với `start`, cả `rest` bị đảo → stack đi từ xa về gần; không có bộ ba nào rẽ phải → giữ đủ mọi điểm. Đúng!
- Chỉ pop khi `< 0` để giữ điểm thẳng hàng.

Vì sao phải đảo tia cuối? Thử bỏ bước đó:

In [13]:
def graham_no_fix(trees):
    pts = [tuple(p) for p in trees]
    start = min(pts, key=lambda p: (p[1], p[0]))
    rest = sorted([p for p in pts if p != start], key=cmp_to_key(
        lambda a, b: (-1 if cross(start, a, b) > 0 else 1) if cross(start, a, b) else dist2(start, a) - dist2(start, b)))
    st = [start]
    for p in rest:
        while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
            st.pop()
        st.append(p)
    return st

sq = [(0, 0), (2, 0), (2, 2), (0, 2), (0, 1)]       # (0,1) nằm trên tia cuối start=(0,0) -> (0,2)
print("không sửa tia cuối:", norm(graham_no_fix(sq)), "| đúng:", norm(outer_trees_brute(sq)))

không sửa tia cuối: [(0, 0), (0, 2), (2, 0), (2, 2)] | đúng: [(0, 0), (0, 1), (0, 2), (2, 0), (2, 2)]


Không đảo thì thứ tự là `... (2,2) → (0,1) → (0,2)`: từ `(2,2)` sang `(0,1)` rồi `(0,2)` là rẽ phải → `(0,1)` bị pop nhầm.

**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionGraham:
    def outerTrees(self, trees: List[List[int]]) -> List[List[int]]:
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

        def d2(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        if len(trees) <= 3:
            return trees
        start = min(trees, key=lambda p: (p[1], p[0]))
        rest = [p for p in trees if p != start]

        def cmp(a, b):
            c = cross(start, a, b)
            return (-1 if c > 0 else 1) if c else d2(start, a) - d2(start, b)

        rest.sort(key=cmp_to_key(cmp))
        i = len(rest) - 1
        while i > 0 and cross(start, rest[-1], rest[i - 1]) == 0:
            i -= 1
        rest[i:] = rest[i:][::-1]
        st = [start]
        for p in rest:
            while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                st.pop()
            st.append(p)
        return st

assert norm(SolutionGraham().outerTrees(trees1)) == norm([[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])
assert norm(SolutionGraham().outerTrees(trees2)) == norm(trees2)
print("SolutionGraham OK")

SolutionGraham OK


**Độ phức tạp Cách 3:** sắp xếp `O(n log n)`, quét stack `O(n)` (mỗi điểm vào/ra stack tối đa một lần) → **`O(n log n)`**, Space `O(n)`.

### 7.3. Cách 4 — Monotone chain (Andrew)

**Giải thích.** Đơn giản hơn Graham vì **không cần sắp theo góc**:

1. Sắp điểm theo `(x, y)`.
2. **Bao dưới (lower hull):** duyệt trái → phải, stack; pop khi `cross(st[-2], st[-1], p) < 0`.
3. **Bao trên (upper hull):** duyệt phải → trái, làm y hệt.
4. Ghép hai bao, bỏ trùng (hai đầu mút, và trường hợp mọi điểm thẳng hàng thì hai bao trùng nhau) bằng `set`.

In [15]:
def outer_trees_monotone(trees):
    pts = sorted(map(tuple, trees))
    if len(pts) <= 1:
        return [list(p) for p in pts]

    def half(points):
        st = []
        for p in points:
            while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                st.pop()
            st.append(p)
        return st

    lower = half(pts)
    upper = half(pts[::-1])
    return [list(p) for p in set(lower + upper)]

print(norm(outer_trees_monotone(trees1)))
assert norm(outer_trees_monotone(trees1)) == norm(outer_trees_brute(trees1))
assert norm(outer_trees_monotone(trees2)) == norm(trees2)

[(1, 1), (2, 0), (2, 4), (3, 3), (4, 2)]


**Đọc code:**

- `half` dùng chung cho cả hai bao — chỉ khác thứ tự duyệt.
- Đi trái → phải mà "chỉ cho rẽ trái" tạo ra **bao dưới**; đi phải → trái tạo **bao trên**.
- `set(lower + upper)`: hai đầu mút (điểm trái nhất, phải nhất) xuất hiện ở cả hai bao; nếu mọi điểm thẳng hàng thì `lower` và `upper` chứa cùng các điểm.

**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionMonotone:
    def outerTrees(self, trees: List[List[int]]) -> List[List[int]]:
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

        pts = sorted(map(tuple, trees))
        if len(pts) <= 1:
            return [list(p) for p in pts]

        def half(points):
            st = []
            for p in points:
                while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                    st.pop()
                st.append(p)
            return st

        return [list(p) for p in set(half(pts) + half(pts[::-1]))]

assert norm(SolutionMonotone().outerTrees(trees1)) == norm([[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])
print("SolutionMonotone OK")

SolutionMonotone OK


**Độ phức tạp Cách 4:** `O(n log n)` cho sắp xếp + `O(n)` cho hai lần quét. Space `O(n)`.

## 8. TRACE từng bước

Trace **Monotone chain** trên ví dụ 1. Sau khi sắp: `(1,1), (2,0), (2,2), (2,4), (3,3), (4,2)`.

**Bao dưới (trái → phải):**

| bước | điểm p | cross(st[-2], st[-1], p) | hành động | stack |
|---|---|---|---|---|
| 1 | (1,1) | — | push | (1,1) |
| 2 | (2,0) | — | push | (1,1),(2,0) |
| 3 | (2,2) | cross((1,1),(2,0),(2,2)) = 2 > 0 | push | (1,1),(2,0),(2,2) |
| 4 | (2,4) | cross((2,0),(2,2),(2,4)) = 0 | push (thẳng hàng, giữ) | ...,(2,2),(2,4) |
| 5 | (3,3) | cross((2,2),(2,4),(3,3)) = -2 < 0 | pop (2,4) | ...,(2,0),(2,2) |
|   |       | cross((2,0),(2,2),(3,3)) = -2 < 0 | pop (2,2) | (1,1),(2,0) |
|   |       | cross((1,1),(2,0),(3,3)) = 4 > 0 | push | (1,1),(2,0),(3,3) |
| 6 | (4,2) | cross((2,0),(3,3),(4,2)) = -4 < 0 | pop (3,3) | (1,1),(2,0) |
|   |       | cross((1,1),(2,0),(4,2)) = 4 > 0 | push | (1,1),(2,0),(4,2) |

Bao trên (phải → trái) tương tự cho `(4,2),(3,3),(2,4),(1,1)`. Ghép lại được 5 điểm.

In [17]:
def trace_monotone(trees):
    pts = sorted(map(tuple, trees))
    print("sau khi sắp:", pts)

    def half(points, name):
        print(f"--- {name} ---")
        st = []
        for p in points:
            while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                print(f"  p={p}: cross({st[-2]}, {st[-1]}, {p}) = {cross(st[-2], st[-1], p)} < 0 -> pop {st[-1]}")
                st.pop()
            if len(st) >= 2:
                print(f"  p={p}: cross({st[-2]}, {st[-1]}, {p}) = {cross(st[-2], st[-1], p)} >= 0 -> push")
            else:
                print(f"  p={p}: stack < 2 phần tử -> push")
            st.append(p)
            print(f"     stack = {st}")
        return st

    lower = half(pts, "bao dưới")
    upper = half(pts[::-1], "bao trên")
    res = sorted(set(lower + upper))
    print("kết quả:", res)
    return res

assert trace_monotone(trees1) == norm([[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])

sau khi sắp: [(1, 1), (2, 0), (2, 2), (2, 4), (3, 3), (4, 2)]
--- bao dưới ---
  p=(1, 1): stack < 2 phần tử -> push
     stack = [(1, 1)]
  p=(2, 0): stack < 2 phần tử -> push
     stack = [(1, 1), (2, 0)]
  p=(2, 2): cross((1, 1), (2, 0), (2, 2)) = 2 >= 0 -> push
     stack = [(1, 1), (2, 0), (2, 2)]
  p=(2, 4): cross((2, 0), (2, 2), (2, 4)) = 0 >= 0 -> push
     stack = [(1, 1), (2, 0), (2, 2), (2, 4)]
  p=(3, 3): cross((2, 2), (2, 4), (3, 3)) = -2 < 0 -> pop (2, 4)
  p=(3, 3): cross((2, 0), (2, 2), (3, 3)) = -2 < 0 -> pop (2, 2)
  p=(3, 3): cross((1, 1), (2, 0), (3, 3)) = 4 >= 0 -> push
     stack = [(1, 1), (2, 0), (3, 3)]
  p=(4, 2): cross((2, 0), (3, 3), (4, 2)) = -4 < 0 -> pop (3, 3)
  p=(4, 2): cross((1, 1), (2, 0), (4, 2)) = 4 >= 0 -> push
     stack = [(1, 1), (2, 0), (4, 2)]
--- bao trên ---
  p=(4, 2): stack < 2 phần tử -> push
     stack = [(4, 2)]
  p=(3, 3): stack < 2 phần tử -> push
     stack = [(4, 2), (3, 3)]
  p=(2, 4): cross((4, 2), (3, 3), (2, 4)) = 0 >= 0 -> pus

Trace **gói quà** trên ví dụ 1: in mỗi cạnh được chọn và các điểm trên cạnh đó.

In [18]:
def trace_jarvis(trees):
    pts = [tuple(p) for p in trees]
    n = len(pts)
    start = min(range(n), key=lambda i: pts[i])
    p, on, step = start, set(), 0
    while True:
        q = (p + 1) % n
        for r in range(n):
            c = cross(pts[p], pts[q], pts[r])
            if c < 0 or (c == 0 and dist2(pts[p], pts[r]) > dist2(pts[p], pts[q])):
                q = r
        seg = sorted(pts[r] for r in range(n) if on_segment(pts[p], pts[q], pts[r]))
        step += 1
        print(f"bước {step}: cạnh {pts[p]} -> {pts[q]}, điểm trên cạnh: {seg}")
        on.update(seg)
        p = q
        if p == start:
            break
    return sorted(on)

assert trace_jarvis(trees1) == norm(outer_trees_brute(trees1))

bước 1: cạnh (1, 1) -> (2, 0), điểm trên cạnh: [(1, 1), (2, 0)]
bước 2: cạnh (2, 0) -> (4, 2), điểm trên cạnh: [(2, 0), (4, 2)]
bước 3: cạnh (4, 2) -> (2, 4), điểm trên cạnh: [(2, 4), (3, 3), (4, 2)]
bước 4: cạnh (2, 4) -> (1, 1), điểm trên cạnh: [(1, 1), (2, 4)]


## 9. Code nộp LeetCode

Chọn Monotone chain: ngắn nhất, `O(n log n)`, không cần sắp theo góc, xử lý tự nhiên cả trường hợp mọi điểm thẳng hàng.

In [19]:
from typing import List

class Solution:
    def outerTrees(self, trees: List[List[int]]) -> List[List[int]]:
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

        pts = sorted(map(tuple, trees))
        if len(pts) <= 1:
            return [list(p) for p in pts]

        def half(points):
            st = []
            for p in points:
                # chỉ bỏ khi RẼ PHẢI thật sự (< 0) để giữ điểm thẳng hàng trên cạnh
                while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                    st.pop()
                st.append(p)
            return st

        hull = set(half(pts)) | set(half(pts[::-1]))   # bao dưới ∪ bao trên
        return [list(p) for p in hull]

sol = Solution()
assert norm(sol.outerTrees(trees1)) == norm([[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])
assert norm(sol.outerTrees(trees2)) == norm([[4, 2], [2, 2], [1, 2]])
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bao dưới của Monotone chain.** Gọi `L` là phần biên bao lồi nhìn từ dưới lên, đi từ điểm trái nhất tới điểm phải nhất (theo thứ tự `(x, y)`).

*Invariant:* sau khi xử lý các điểm `p_1..p_i` (đã sắp), stack chứa đúng **bao dưới (kể cả điểm thẳng hàng trên cạnh) của tập `{p_1..p_i}`**, theo thứ tự trái → phải, và mọi bộ ba liên tiếp có `cross >= 0`.

- Thêm `p_{i+1}` (có `x` lớn nhất cho tới giờ): `p_{i+1}` chắc chắn thuộc bao dưới mới (nó là điểm phải nhất). Một điểm `s` trên đỉnh stack bị loại khỏi bao dưới ⇔ nó nằm **phía trên** đoạn nối điểm dưới nó với `p_{i+1}` ⇔ `cross(st[-2], s, p_{i+1}) < 0`. Pop lặp lại cho tới khi rẽ trái hoặc thẳng hàng → stack lại là bao dưới.
- Điểm thẳng hàng (`cross = 0`) nằm **trên** cạnh nên vẫn thuộc biên → không pop.

Bao trên đối xứng. Hợp hai bao là toàn bộ biên. ∎

**Gói quà.** Ở mỗi bước, `q` được chọn sao cho không điểm nào nằm bên phải tia `p → q` → `pq` là đường thẳng tựa, nên là cạnh của bao lồi; chọn `q` xa nhất đảm bảo `q` là đỉnh. Bắt đầu từ một đỉnh và đi tới đỉnh kế tiếp ngược chiều kim đồng hồ, thuật toán đi đủ một vòng biên rồi dừng ở `start`.

**Graham.** Sau khi sắp theo góc, các điểm lần lượt nằm theo thứ tự quét quanh `start`; stack giữ bao lồi (biên) của tập đã quét, lập luận giống bao dưới. Đảo tia cuối đảm bảo các điểm trên cạnh khép kín về `start` theo đúng thứ tự trên biên.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n^3)` | `O(n)` | chỉ để kiểm chứng |
| 2. Gói quà | `O(n · h)` | `O(n)` | nhanh khi ít đỉnh |
| 3. Graham | `O(n log n)` | `O(n)` | sắp theo góc, cần sửa tia cuối |
| 4. Monotone chain | `O(n log n)` | `O(n)` | **nộp bài** |

Đo trên `n = 3000` điểm ngẫu nhiên trong lưới `101 × 101`:

In [20]:
random.seed(1)
big = [list(p) for p in random.sample([(x, y) for x in range(101) for y in range(101)], 3000)]
for name, cls in [("Jarvis", SolutionJarvis), ("Graham", SolutionGraham), ("Monotone", SolutionMonotone)]:
    t = time.perf_counter()
    r = cls().outerTrees([p[:] for p in big])
    print(f"{name:>9}: {len(r)} điểm trên hàng rào, {time.perf_counter() - t:.3f}s")

   Jarvis: 98 điểm trên hàng rào, 0.007s
   Graham: 98 điểm trên hàng rào, 0.006s
 Monotone: 98 điểm trên hàng rào, 0.003s


## 12. Bộ test

In [21]:
impls = {
    "jarvis": outer_trees_jarvis,
    "graham": outer_trees_graham,
    "monotone": outer_trees_monotone,
    "SolutionJarvis": lambda t: SolutionJarvis().outerTrees(t),
    "SolutionGraham": lambda t: SolutionGraham().outerTrees(t),
    "SolutionMonotone": lambda t: SolutionMonotone().outerTrees(t),
    "Solution": lambda t: Solution().outerTrees(t),
}

def check(trees, expected=None):
    exp = norm(expected) if expected is not None else norm(outer_trees_brute(trees))
    for name, f in impls.items():
        got = norm(f([p[:] for p in trees]))
        assert got == exp, (name, trees, got, exp)

# Ví dụ đề
check(trees1, [[1, 1], [2, 0], [4, 2], [3, 3], [2, 4]])
check(trees2, [[4, 2], [2, 2], [1, 2]])

# Edge case
check([[0, 0]], [[0, 0]])                                         # 1 điểm
check([[0, 0], [5, 5]], [[0, 0], [5, 5]])                         # 2 điểm
check([[0, 0], [1, 1], [2, 2], [3, 3], [4, 4]])                   # thẳng hàng chéo
check([[0, 3], [0, 0], [0, 1], [0, 2]])                           # thẳng hàng dọc
check([[0, 0], [2, 0], [2, 2], [0, 2], [1, 0], [2, 1], [1, 2], [0, 1], [1, 1]],
      [[0, 0], [2, 0], [2, 2], [0, 2], [1, 0], [2, 1], [1, 2], [0, 1]])   # hình vuông có điểm giữa cạnh
check([[0, 0], [4, 0], [0, 4], [1, 1], [2, 2], [1, 3]])            # điểm trên cạnh huyền
print("Ví dụ đề + edge case OK")

# Ngẫu nhiên có seed trên lưới nhỏ (nhiều điểm thẳng hàng)
random.seed(587)
for _ in range(400):
    size = random.randint(2, 6)
    n = random.randint(1, min(20, size * size))
    pts = [list(p) for p in random.sample([(x, y) for x in range(size) for y in range(size)], n)]
    check(pts)
print("400 test ngẫu nhiên khớp brute force")

# Lưới lớn hơn: so các cách nhanh với nhau
for _ in range(30):
    pts = [list(p) for p in random.sample([(x, y) for x in range(40) for y in range(40)], random.randint(50, 300))]
    ref = norm(Solution().outerTrees(pts))
    assert all(norm(f([p[:] for p in pts])) == ref for f in impls.values())
print("30 test lớn: các cách khớp nhau")

Ví dụ đề + edge case OK
400 test ngẫu nhiên khớp brute force


30 test lớn: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Pop cả khi thẳng hàng (`<= 0`)

In [22]:
def wrong_pop_collinear(trees):
    pts = sorted(map(tuple, trees))
    if len(pts) <= 1:
        return pts
    def half(points):
        st = []
        for p in points:
            while len(st) >= 2 and cross(st[-2], st[-1], p) <= 0:    # SAI cho bài này
                st.pop()
            st.append(p)
        return st
    return list(set(half(pts) + half(pts[::-1])))

print("sai:", norm(wrong_pop_collinear(trees1)))
print("đúng:", norm(Solution().outerTrees(trees1)))

sai: [(1, 1), (2, 0), (2, 4), (4, 2)]
đúng: [(1, 1), (2, 0), (2, 4), (3, 3), (4, 2)]


**Vì sao sai:** `<= 0` là phiên bản sách giáo khoa chỉ lấy **đỉnh**. Đề này muốn cả cây nằm giữa cạnh, ví dụ `(3,3)`.

**Code đúng:** pop chỉ khi `< 0`.

### Lỗi 2 — Không khử trùng lặp khi ghép bao dưới và bao trên

In [23]:
def wrong_no_dedup(trees):
    pts = sorted(map(tuple, trees))
    def half(points):
        st = []
        for p in points:
            while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                st.pop()
            st.append(p)
        return st
    return half(pts) + half(pts[::-1])

print("số điểm trả về (sai):", len(wrong_no_dedup(trees2)), "| đúng:", len(Solution().outerTrees(trees2)))

số điểm trả về (sai): 6 | đúng: 3


**Vì sao sai:** hai đầu mút luôn thuộc cả hai bao; khi mọi điểm thẳng hàng, **mọi** điểm thuộc cả hai bao → trả về trùng lặp, LeetCode báo sai.

**Code đúng:** dùng `set` (hoặc cắt `lower[:-1] + upper[:-1]` rồi vẫn cần khử trùng cho trường hợp thẳng hàng).

### Lỗi 3 — Dùng góc `atan2` (số thực) để sắp xếp

In [24]:
import math
a, b = (1, 3), (2, 6)          # cùng tia từ gốc
print(math.atan2(3, 1) == math.atan2(6, 2), math.atan2(3, 1), math.atan2(6, 2))
o = (0, 0)
p, q = (10**17, 1), (10**17 + 1, 1)          # KHÔNG cùng tia (cross = -1)
print("atan2 bằng nhau:", math.atan2(1, 10**17) == math.atan2(1, 10**17 + 1), "| cross:", cross(o, p, q))

True 1.2490457723982544 1.2490457723982544
atan2 bằng nhau: True | cross: -1


**Vì sao nguy hiểm:** số thực có sai số làm tròn; hai điểm khác góc có thể cho `atan2` bằng nhau (và ngược lại, hai điểm cùng góc có thể lệch nhau ở chữ số cuối). Với điểm thẳng hàng — trọng tâm của bài này — sai một chút là pop nhầm.

**Code đúng:** so sánh bằng tích có hướng (số nguyên, chính xác tuyệt đối).

### Lỗi 4 — Graham scan quên đảo tia cuối

Đã minh họa ở mục 7.2 (`graham_no_fix` làm mất điểm `(0,1)`).

**Code đúng:** sau khi sắp, đảo đoạn cuối các điểm thẳng hàng với `start`.

### Lỗi 5 — Trả về `list` của `tuple` hoặc quên trường hợp 1 điểm

Monotone chain với 1 điểm: `half` trả `[p]` cho cả hai bao — vẫn đúng; nhưng nhiều cài đặt dùng `lower[:-1] + upper[:-1]` sẽ trả **rỗng** với 1 điểm. Luôn thử `[[0,0]]`.

## 14. Mẹo & nhận dạng pattern

- **"Sợi dây/hàng rào ngắn nhất bao mọi điểm"**, "đa giác lồi nhỏ nhất chứa các điểm", "điểm xa nhất" → **bao lồi**.
- **Tích có hướng** là phép toán nền của mọi bài hình học có tọa độ nguyên: hướng rẽ, điểm trong đa giác, giao đoạn thẳng, diện tích (công thức shoelace).
- **Có / không giữ điểm thẳng hàng** chỉ khác nhau ở `< 0` hay `<= 0` — đọc đề kỹ.
- **Monotone chain** là lựa chọn mặc định khi thi: ngắn, ổn định, không cần góc.
- **Test trên lưới nhỏ** (tọa độ 0..5) để sinh nhiều trường hợp thẳng hàng — nơi bug hay trốn.

## 15. Biến thể

1. **Chỉ lấy đỉnh của bao lồi** (bỏ điểm giữa cạnh): pop khi `<= 0`.
2. **Chu vi / diện tích hàng rào**: duyệt đỉnh theo thứ tự, cộng độ dài cạnh / dùng công thức shoelace (Bài tập 5).
3. **Kiểm tra đa giác lồi (LeetCode 469 Convex Polygon, Premium):** mọi `cross` của ba đỉnh liên tiếp cùng dấu.
4. **Điểm xa nhất (đường kính tập điểm):** bao lồi + "rotating calipers".
5. **Largest Triangle Area (LeetCode 812):** đỉnh tam giác lớn nhất nằm trên bao lồi.

In [25]:
# Biến thể 1: chỉ lấy đỉnh (strict hull) — trả theo thứ tự ngược chiều kim đồng hồ
def hull_vertices(points):
    pts = sorted(set(map(tuple, points)))
    if len(pts) <= 2:
        return pts
    def half(seq):
        st = []
        for p in seq:
            while len(st) >= 2 and cross(st[-2], st[-1], p) <= 0:
                st.pop()
            st.append(p)
        return st
    lower, upper = half(pts), half(pts[::-1])
    return lower[:-1] + upper[:-1]

print("đỉnh bao lồi ví dụ 1:", hull_vertices(trees1))
assert sorted(hull_vertices(trees1)) == [(1, 1), (2, 0), (2, 4), (4, 2)]

đỉnh bao lồi ví dụ 1: [(1, 1), (2, 0), (4, 2), (2, 4)]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | cặp `PQ` có mọi điểm cùng phía → thêm điểm trên đoạn | `O(n^3)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Gói quà | từ đỉnh trái nhất, mỗi bước chọn điểm "ngoài cùng" | `O(n · h)` | `O(n)` | bao lồi ít đỉnh |
| 3. Graham | sắp theo góc quanh điểm thấp nhất + stack | `O(n log n)` | `O(n)` | kinh điển, cần cẩn thận tia cuối |
| 4. Monotone chain | sắp theo `(x, y)`, bao dưới + bao trên | `O(n log n)` | `O(n)` | **nộp bài**, mặc định |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: kết quả (dưới dạng tập) cho `[[0,0],[1,0],[2,0],[1,1]]`?

**Bài 2 (tính tay).** Tính `cross((0,0), (4,0), (2,3))`, `cross((4,0), (0,4), (2,2))`, `cross((0,4), (0,0), (1,1))` và cho biết mỗi điểm thứ ba nằm bên nào.

**Bài 3 (điền TODO).** Hoàn thiện hàm `is_inside_strict(hull_ccw, p)` kiểm tra `p` nằm **hẳn bên trong** đa giác lồi cho bởi danh sách đỉnh theo chiều ngược kim đồng hồ.

**Bài 4 (tự viết lại).** Tự viết lại Monotone chain mà không nhìn code (hàm `my_outer_trees`).

**Bài 5 (mini challenge).** Viết `fence_length_and_area(trees)` trả về `(chu vi, diện tích)` của hàng rào (dùng đỉnh bao lồi theo thứ tự, công thức shoelace). Với tập thẳng hàng, diện tích bằng 0 và chu vi bằng **hai lần** độ dài đoạn.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def is_inside_strict(hull_ccw, p):
    # TODO: với mọi cạnh (hull[i], hull[i+1]), cross(...) phải > 0
    pass

# Bài 4 — TODO
def my_outer_trees(trees):
    # TODO: sort, bao dưới, bao trên, pop khi cross < 0, khử trùng lặp
    pass

# Bài 5 — TODO
def fence_length_and_area(trees):
    # TODO: lấy đỉnh bao lồi theo thứ tự (hull_vertices), cộng độ dài cạnh, shoelace / 2
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `(1,1)` nằm trên hay trong tam giác?
- Mức 2: tam giác `(0,0),(2,0),(1,1)`; `(1,0)` nằm giữa cạnh đáy.
- Mức 3: cả 4 điểm.

**Bài 2.**
- Mức 1: áp công thức `(a.x-o.x)(b.y-o.y) - (a.y-o.y)(b.x-o.x)`.
- Mức 2: kết quả lần lượt `12`, `0`, `...`.
- Mức 3: dương = trái, 0 = thẳng hàng.

**Bài 3.**
- Mức 1: đa giác lồi, đi ngược chiều kim đồng hồ: điểm bên trong luôn ở bên trái mọi cạnh.
- Mức 2: dùng `zip(hull, hull[1:] + hull[:1])`.
- Mức 3: `all(cross(a, b, p) > 0 for a, b in ...)`.

**Bài 4.**
- Mức 1: `pts = sorted(map(tuple, trees))`.
- Mức 2: một hàm `half` dùng cho cả hai chiều.
- Mức 3: trả `[list(p) for p in set(lower + upper)]`.

**Bài 5.**
- Mức 1: dùng `hull_vertices` (đỉnh theo thứ tự ngược chiều kim đồng hồ).
- Mức 2: chu vi = `Σ dist(v[i], v[i+1])` (vòng kín), diện tích = `|Σ cross(0, v[i], v[i+1])| / 2`.
- Mức 3: 2 đỉnh (thẳng hàng) → chu vi `2 · dist`, diện tích 0; 1 đỉnh → `(0, 0)`.

## 19. Đáp án tham khảo

**Bài 1:** cả 4 điểm `{(0,0),(1,0),(2,0),(1,1)}`.

**Bài 2:** `12` (trái), `0` (thẳng hàng — `(2,2)` nằm trên cạnh `(4,0)–(0,4)`), `cross((0,4),(0,0),(1,1)) = (0)(1-4) - (-4)(1-0) = 4` (trái).

In [27]:
assert norm(Solution().outerTrees([[0, 0], [1, 0], [2, 0], [1, 1]])) == [(0, 0), (1, 0), (1, 1), (2, 0)]
assert cross((0, 0), (4, 0), (2, 3)) == 12
assert cross((4, 0), (0, 4), (2, 2)) == 0
assert cross((0, 4), (0, 0), (1, 1)) == 4

# Bài 3
def is_inside_strict(hull_ccw, p):
    return all(cross(a, b, p) > 0 for a, b in zip(hull_ccw, hull_ccw[1:] + hull_ccw[:1]))

hv = hull_vertices(trees1)
print("đỉnh:", hv, "| (2,2) bên trong?", is_inside_strict(hv, (2, 2)), "| (3,3)?", is_inside_strict(hv, (3, 3)))
assert is_inside_strict(hv, (2, 2)) and not is_inside_strict(hv, (3, 3))

# Bài 4
def my_outer_trees(trees):
    pts = sorted(map(tuple, trees))
    def half(seq):
        st = []
        for p in seq:
            while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:
                st.pop()
            st.append(p)
        return st
    return [list(p) for p in set(half(pts) + half(pts[::-1]))]

random.seed(4)
for _ in range(200):
    pts = [list(p) for p in random.sample([(x, y) for x in range(6) for y in range(6)], random.randint(1, 15))]
    assert norm(my_outer_trees(pts)) == norm(outer_trees_brute(pts))
print("Bài 3, 4 OK")

đỉnh: [(1, 1), (2, 0), (4, 2), (2, 4)] | (2,2) bên trong? True | (3,3)? False
Bài 3, 4 OK


In [28]:
# Bài 5
def fence_length_and_area(trees):
    v = hull_vertices(trees)
    if len(v) == 1:
        return 0.0, 0.0
    per = sum(math.dist(v[i], v[(i + 1) % len(v)]) for i in range(len(v)))
    area2 = sum(cross((0, 0), v[i], v[(i + 1) % len(v)]) for i in range(len(v)))
    return per, abs(area2) / 2

print(fence_length_and_area(trees1))
assert fence_length_and_area([[0, 0], [3, 0], [3, 4], [0, 4], [1, 1]]) == (14.0, 12.0)
assert fence_length_and_area([[0, 0], [1, 0], [3, 0]]) == (6.0, 0.0)
assert fence_length_and_area([[5, 5]]) == (0.0, 0.0)
p, a = fence_length_and_area(trees1)
assert abs(a - 6.0) < 1e-9          # tứ giác (1,1),(2,0),(4,2),(2,4)
print("Bài 5 OK")

(10.233345472033855, 6.0)
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Monotone chain (giữ điểm thẳng hàng):**

```python
def cross(o, a, b):
    return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])

pts = sorted(map(tuple, trees))
def half(seq):
    st = []
    for p in seq:
        while len(st) >= 2 and cross(st[-2], st[-1], p) < 0:   # <= 0 nếu chỉ lấy đỉnh
            st.pop()
        st.append(p)
    return st
hull = set(half(pts)) | set(half(pts[::-1]))
```

**Câu thần chú:** *"Sắp theo x, đi dưới rồi đi trên, rẽ phải thì bỏ, thẳng hàng thì giữ."*

**Checklist trước khi submit:**

- [ ] Dùng tích có hướng số nguyên, không dùng góc số thực.
- [ ] Điều kiện pop đúng: `< 0` (giữ điểm thẳng hàng).
- [ ] Khử trùng lặp khi ghép hai bao.
- [ ] Thử 1 điểm, 2 điểm, mọi điểm thẳng hàng (ngang, dọc, chéo).
- [ ] Thử hình vuông có điểm giữa mỗi cạnh và điểm ở tâm.
- [ ] Trả về `list` các `[x, y]`.
- [ ] Đối chiếu brute force trên lưới nhỏ.